# Facial Attributes with GCN (Graph Convolutional Networks)

Notebook này minh họa hướng **nghiên cứu**: thay vì học trực tiếp từ pixel (CNN), ta sẽ:
1. Lấy ảnh khuôn mặt (từ UTKFace/FairFace/FER2013)
2. Trích **landmarks** (ví dụ 68 điểm) → mỗi điểm là **node** của đồ thị
3. Xây **đồ thị khuôn mặt**: node = landmark, edge = kết nối lân cận / bộ phận (mắt, mũi, miệng)
4. Đưa đồ thị vào **GCN** để dự đoán: tuổi theo nhóm, giới tính, chủng tộc, (và có thể emotion)

Vì môi trường có thể chưa cài `torch-geometric`, mình sẽ để sẵn **cell cài đặt**.

Bài này phù hợp để 1 bạn trong nhóm làm hướng *"mô hình dựa trên cấu trúc khuôn mặt"* khác hẳn 4 bạn dùng CNN.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [4]:
path = r'/content/drive/MyDrive/Nhom1_Py/Dataset'
os.listdir(path)

['utkface-new', 'fer2013', 'sample_img', 'ex_mobilenet', 'test_vgg']

## 2. Cài đặt thư viện cần thiết

GCN trong PyTorch thường dùng **PyTorch Geometric**.
Trên Colab, bạn có thể chạy:
```bash
pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.3.0+cpu.html
```

> Nếu bạn không cài được, vẫn có thể chạy ví dụ với `networkx` + GCN tự code, nhưng tốt nhất là dùng `torch_geometric`.

In [5]:
!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.3.0+cpu.html

Looking in links: https://data.pyg.org/whl/torch-2.3.0+cpu.html


In [6]:
pip install mediapipe torch-geometric opencv-python

In [ ]:
# 2.1. Import cơ bản
import os, glob
from PIL import Image
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# thử import torch_geometric (nếu chưa có sẽ báo lỗi, bạn cài theo cell markdown phía trên)
try:
    from torch_geometric.data import Data, Dataset as GeoDataset, DataLoader as GeoDataLoader
    from torch_geometric.nn import GCNConv, global_mean_pool
    HAS_PYG = True
except Exception as e:
    print("⚠️ Chưa cài torch_geometric. Hãy cài lên trước khi train.")
    HAS_PYG = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cpu.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_cluster/_version_cpu.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cpu.so
  import torch_geometric.typing


## 3. Ý tưởng tạo graph từ UTKFace

Vì UTKFace không cung cấp sẵn landmarks, nên có 2 cách:
1. **C1 (đúng chuẩn)**: chạy thêm bước detect + landmark (Mediapipe / dlib) → lưu ra `.npy` → load vào GCN. (dùng khi làm báo cáo thật)
2. **C2 (demo trong notebook)**: tạo **giả landmarks** bằng cách chia ảnh 224×224 thành 68 điểm cố định (grid) → vẫn tạo được đồ thị để chạy GCN → phù hợp demo train code.

Ở đây mình làm theo C2 để notebook chạy được luôn. Khi bạn có landmarks thật thì chỉ cần thay bước `fake_landmarks(...)`.

In [8]:
import cv2
import mediapipe as mp
import torch
import numpy as np

# Khởi tạo MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,      # Chế độ ảnh tĩnh (chính xác hơn cho training)
    max_num_faces=1,             # Chỉ lấy 1 mặt
    refine_landmarks=True,       # Lấy thêm điểm ở mắt/môi cho chi tiết
    min_detection_confidence=0.5
)

def get_real_landmarks(image_path):
    """
    Đầu vào: Đường dẫn ảnh
    Đầu ra: Tensor (478, 2) chứa tọa độ x, y của các điểm trên mặt
    """
    # 1. Đọc ảnh
    img = cv2.imread(image_path)
    if img is None:
        return None # Ảnh lỗi

    # MediaPipe cần ảnh RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 2. Trích xuất landmark
    results = face_mesh.process(img_rgb)

    # 3. Xử lý kết quả
    if results.multi_face_landmarks:
        # Lấy khuôn mặt đầu tiên tìm thấy
        face_landmarks = results.multi_face_landmarks[0]

        # Chuyển đổi thành list tọa độ [x, y]
        # MediaPipe trả về tọa độ tỉ lệ (0.0 -> 1.0), rất tốt cho GCN (không cần scale lại)
        landmarks = []
        for lm in face_landmarks.landmark:
            landmarks.append([lm.x, lm.y]) # Chỉ lấy x, y (bỏ z)

        return torch.tensor(landmarks, dtype=torch.float)
    else:
        return None # Không tìm thấy mặt trong ảnh

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


## 4. Dataset chuyên cho GCN

Dataset này sẽ:
1. Đọc ảnh UTKFace → (nếu có landmarks thật thì load, còn không thì tạo giả)
2. Tạo **edge_index** theo kiểu k-nearest-neighbor (ví dụ k=4) trên 68 điểm
3. Trả về `torch_geometric.data.Data` gồm:
   - `x`: node features (68, 2) – tọa độ landmark
   - `edge_index`: (2, E)
   - `y_age`, `y_gender`, `y_race`


In [ ]:
def build_knn_graph(points, k=4):
    """points: tensor (N, 2)
    return edge_index (2, E)
    """
    N = points.size(0)
    dists = torch.cdist(points, points)  # (N, N)
    edge_index = []
    for i in range(N):
        # lấy k+1 điểm gần nhất (bao gồm chính nó) rồi bỏ chính nó đi
        _, idx = torch.topk(dists[i], k+1, largest=False)
        for j in idx[1:]:  # bỏ chính nó
            edge_index.append([i, j.item()])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    return edge_index

def age_to_group5(age: int) -> int:
    """
    Chia tuổi thành 5 nhóm:
    0: 0-18
    1: 19-30
    2: 31-45
    3: 46-60
    4: 61+
    """
    if age <= 18:
        return 0
    elif age <= 30:
        return 1
    elif age <= 45:
        return 2
    elif age <= 60:
        return 3
    else:
        return 4

class UTKFaceGraphDataset(Dataset):
    def __init__(self, root_dir, num_points=68, k=4):
        self.root_dir = root_dir
        self.num_points = num_points
        self.k = k

        self.image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            self.image_paths.extend(glob.glob(os.path.join(root_dir, ext)))
        self.image_paths.sort()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        filename = os.path.basename(img_path)
        # UTKFace: age_gender_race_date.jpg
        parts = filename.split('_')
        age = int(parts[0])
        gender = int(parts[1])
        race = int(parts[2])
        # --- dùng hàm mới để phân nhóm tuổi ---
        age_cls = age_to_group5(age)
        # --- tạo landmarks giả ---
        points = fake_landmarks(self.num_points)  # (N, 2)
        edge_index = build_knn_graph(points, k=self.k)

        if HAS_PYG:
            data = Data(x=points, edge_index=edge_index)
            data.y_age = torch.tensor([age_cls], dtype=torch.long)
            data.y_gender = torch.tensor([gender], dtype=torch.long)
            data.y_race = torch.tensor([race], dtype=torch.long)
            return data
        else:
            # nếu không có torch_geometric, trả về dict để debug
            return {
                'x': points,
                'edge_index': edge_index,
                'age_cls': age_cls,
                'gender': gender,
                'race': race
            }

In [ ]:
# 4.1. Tạo DataLoader
train_dir = "/content/drive/MyDrive/Nhom1_Py/Dataset/utkface-new/train"
val_dir = "/content/drive/MyDrive/Nhom1_Py/Dataset/utkface-new/val"

if os.path.exists(train_dir):
    train_graph_ds = UTKFaceGraphDataset(train_dir)
    if HAS_PYG:
        train_loader = GeoDataLoader(train_graph_ds, batch_size=64, shuffle=True)
    else:
        train_loader = DataLoader(train_graph_ds, batch_size=64, shuffle=True)
else:
    train_graph_ds = None
    train_loader = None

if os.path.exists(val_dir):
    val_graph_ds = UTKFaceGraphDataset(val_dir)
    if HAS_PYG:
        val_loader = GeoDataLoader(val_graph_ds, batch_size=64, shuffle=False)
    else:
        val_loader = DataLoader(val_graph_ds, batch_size=64, shuffle=False)
else:
    val_graph_ds = None
    val_loader = None

train_graph_ds

## 5. Model GCN đa head

Kiến trúc:
- 2 lớp GCNConv
- Global mean pooling
- FC shared
- 3 head: age (5), gender (2), race (5)

Nếu bạn muốn thêm **emotion** thì chỉ cần thêm 1 head nữa.

In [ ]:
if HAS_PYG:
    class FaceGCN(nn.Module):
        def __init__(self, num_node_features=2, num_age=5, num_gender=2, num_race=5):
            super().__init__()
            self.conv1 = GCNConv(num_node_features, 64)
            self.conv2 = GCNConv(64, 128)
            self.fc_shared = nn.Linear(128, 64)
            self.age_head = nn.Linear(64, num_age)
            self.gender_head = nn.Linear(64, num_gender)
            self.race_head = nn.Linear(64, num_race)

        def forward(self, data):
            x, edge_index, batch = data.x, data.edge_index, data.batch
            x = torch.relu(self.conv1(x, edge_index))
            x = torch.relu(self.conv2(x, edge_index))
            x = global_mean_pool(x, batch)
            x = torch.relu(self.fc_shared(x))
            age_logits = self.age_head(x)
            gender_logits = self.gender_head(x)
            race_logits = self.race_head(x)
            return age_logits, gender_logits, race_logits

    model = FaceGCN().to(device)
else:
    model = None
model

## 6. Hàm train / eval 5 epoch

Tương tự các notebook trước: tổng loss = loss_age + loss_gender + loss_race.

In [ ]:
def train_one_epoch(model, loader, opt, ce_age, ce_gender, ce_race):
    model.train()
    run_loss = 0.0
    for data in loader:
        data = data.to(device)
        opt.zero_grad()
        age_logits, gender_logits, race_logits = model(data)
        loss_age = ce_age(age_logits, data.y_age)
        loss_gender = ce_gender(gender_logits, data.y_gender)
        loss_race = ce_race(race_logits, data.y_race)
        loss = loss_age + loss_gender + loss_race
        loss.backward()
        opt.step()
        run_loss += loss.item() * data.num_graphs
    return run_loss / len(loader.dataset)

In [ ]:
@torch.no_grad()
def eval_one_epoch(model, loader, ce_age, ce_gender, ce_race):
    model.eval()
    total = 0
    run_loss = 0.0
    correct_age = 0
    correct_gender = 0
    correct_race = 0
    for data in loader:
        data = data.to(device)
        age_logits, gender_logits, race_logits = model(data)
        loss_age = ce_age(age_logits, data.y_age)
        loss_gender = ce_gender(gender_logits, data.y_gender)
        loss_race = ce_race(race_logits, data.y_race)
        loss = loss_age + loss_gender + loss_race
        run_loss += loss.item() * data.num_graphs

        _, age_pred = age_logits.max(1)
        _, gender_pred = gender_logits.max(1)
        _, race_pred = race_logits.max(1)
        correct_age += (age_pred == data.y_age).sum().item()
        correct_gender += (gender_pred == data.y_gender).sum().item()
        correct_race += (race_pred == data.y_race).sum().item()
        total += data.num_graphs
    avg_loss = run_loss / len(loader.dataset)
    age_acc = correct_age / total
    gender_acc = correct_gender / total
    race_acc = correct_race / total
    return avg_loss, age_acc, gender_acc, race_acc

In [ ]:
# 7. Train 5 epoch để tìm model tốt nhất
num_epochs = 5

if model is not None and train_loader is not None and val_loader is not None:
    ce_age = nn.CrossEntropyLoss()
    ce_gender = nn.CrossEntropyLoss()
    ce_race = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_val_loss = float('inf')
    best_state = None
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, opt, ce_age, ce_gender, ce_race)
        val_loss, age_acc, gender_acc, race_acc = eval_one_epoch(model, val_loader, ce_age, ce_gender, ce_race)
        print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | age_acc={age_acc:.3f} | gender_acc={gender_acc:.3f} | race_acc={race_acc:.3f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
else:
    print("⚠️ Chưa sẵn sàng train: cần cài torch-geometric và có data trong data/utkface/... ")

In [ ]:
# 8. Lưu model tốt nhất
if 'best_state' in locals() and best_state is not None:
    os.makedirs('checkpoints', exist_ok=True)
    torch.save(best_state, 'checkpoints/gcn_face_multihead_best.pth')
    print('Đã lưu checkpoints/gcn_face_multihead_best.pth')
else:
    print('Không có model để lưu (có thể chưa train do thiếu data hoặc chưa cài torch-geometric).')

In [ ]:
!cp "/content/checkpoints/gcn_face_multihead_best.pth" "/content/drive/MyDrive/Nhom1_Py/"

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
import glob
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1) DEFINE MODEL
class FaceGCN(nn.Module):
    def __init__(self, num_node_features=2, num_age=5, num_gender=2, num_race=5):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, 64)
        self.conv2 = GCNConv(64, 128)
        self.fc_shared = nn.Linear(128, 64)
        self.age_head = nn.Linear(64, num_age)
        self.gender_head = nn.Linear(64, num_gender)
        self.race_head = nn.Linear(64, num_race)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        x = torch.relu(self.fc_shared(x))
        return self.age_head(x), self.gender_head(x), self.race_head(x)

# Load model
model = FaceGCN(
    num_node_features=2,
    num_age=5,
    num_gender=2,
    num_race=5
).to(device)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/Nhom1_Py/gcn_face_multihead_best.pth", map_location=device)
)
model.eval()

print("✔ Model loaded!")

In [ ]:
# 2) FAKE LANDMARKS + GRAPH BUILD
import numpy as np

def fake_landmarks(n=68):
    """Fake landmark generator (vì bạn chưa có landmark detector)."""
    pts = np.random.rand(n, 2).astype(np.float32)
    return torch.tensor(pts, dtype=torch.float)

def build_knn_graph(points, k=4):
    """Xây dựng KNN graph cho landmark."""
    from sklearn.neighbors import NearestNeighbors
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='ball_tree').fit(points)
    distances, indices = nbrs.kneighbors(points)

    edge_list = []
    for i in range(points.shape[0]):
        for j in indices[i][1:]:  # bỏ chính nó
            edge_list.append([i, j])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return edge_index

In [ ]:
# 3) LOAD TEST IMAGES
test_dir = "/content/drive/MyDrive/Nhom1_Py/Dataset/sample_img"

test_images = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    test_images.extend(glob.glob(f"{test_dir}/{ext}"))

test_images.sort()

print("📌 Số lượng ảnh test:", len(test_images))

In [ ]:
# 4) RUN PREDICTION FOR EACH IMAGE
age_groups = ["0-18", "19-30", "31-45", "46-60", "61+"]
gender_map = ["Female", "Male"]
race_map   = ["White", "Black", "Asian", "Indian", "Others"]

for img_path in test_images:
    print("\n Đang test ảnh:", img_path)

    img = Image.open(img_path).convert("RGB")  # bạn có thể resize nếu muốn

    # ---- Tạo landmark & graph ----
    points = fake_landmarks(68)
    edge_index = build_knn_graph(points, k=4)

    data = Data(x=points, edge_index=edge_index)
    data.batch = torch.zeros(points.shape[0], dtype=torch.long)   # batch = 0 cho toàn graph
    data = data.to(device)

    # ---- Predict ----
    with torch.no_grad():
        out_age, out_gender, out_race = model(data)

    pred_age = out_age.argmax(dim=1).item()
    pred_gender = out_gender.argmax(dim=1).item()
    pred_race = out_race.argmax(dim=1).item()

    print("  Age group:", age_groups[pred_age])
    print("  Gender   :", gender_map[pred_gender])
    print("  Race     :", race_map[pred_race])
